## Feature Engineering — Nigerian Credit Risk Pipeline

### Objective
Transform cleaned loan data into a rich feature set that simulates the 
feature engineering process used by Nigerian fintech lenders. Features 
are engineered in SQL to reflect production data warehouse practices.

### Data Source Mapping
| Lending Club Feature | Nigerian Fintech Equivalent |
|---|---|
| `annual_inc` | Salary data from payroll APIs (Mono, Okra) |
| `revol_util` | Credit bureau utilization (CRC, FirstCentral) |
| `inq_last_6mths` | Bureau inquiry count |
| `delinq_2yrs` | Bureau delinquency history |
| `dti` | Computed from bank statement analysis |
| `verification_status` | IPPIS/payroll verification |

### Features Created
- **Affordability (5):** income_to_loan_ratio, payment_coverage_ratio, 
  overall_leverage_ratio, cbn_dti_compliant, high_dti_flag
- **Utilization (1):** revol_util_bucket
- **Delinquency (5):** has_delinquency_2yrs, has_pub_rec, has_bankruptcy, 
  has_current_delinq, risk_flag_count
- **Inquiry (3):** high_inquiry_flag, recent_inquiry_flag, inquiry_rate
- **Credit Maturity (3):** credit_age_bucket, credit_breadth, thin_file_flag
- **Income (2):** income_segment, income_verified_flag
- **Revolving Health (3):** revol_utilization_ratio, 
  avg_credit_limit_per_account, pct_maxed_cards

### Key Decisions & Rationale
- **Dropped grade/sub_grade:** Pre-assigned risk scores not available 
  in a real Nigerian fintech pipeline — including them would cause leakage
- **Adjusted credit_age thresholds:** Original thresholds (<5, 5-10, >10) 
  designed for Nigerian thin-file borrowers. Lending Club data shows all 
  borrowers have credit_age > 10 years (min: 10.76, median: 25.7), so 
  thresholds were adjusted to reflect actual distribution
- **Excluded bc_to_revol_ratio:** bc_util and revol_util are likely highly 
  correlated — the ratio adds noise rather than signal
- **Nulls in overall_leverage_ratio (3,132 rows):** Caused by tot_hi_cred_lim = 0. 
  Will be handled via median imputation in the modelling pipeline

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

In [2]:
# Create a connection
load_dotenv()
engine = create_engine(os.getenv('DATABASE_URL'))

In [3]:
# Run SQL file
path = '../sql/feature_engineering.sql'
with open(path, 'r') as f:
    sql_script = f.read()

# Execute SQL file
with engine.connect() as conn:
    conn.execute(text(sql_script))
    conn.commit()

print("SQL file executed!")

SQL file executed!


In [4]:
# Convert to DataFrame
loans_features = pd.read_sql("SELECT * FROM loan_features", engine)

# Save CSV
loans_features.to_csv("../data/features/loan_features.csv", index=False)

print(f"Saved {loans_features.shape[0]} rows, {loans_features.shape[1]} columns")

Saved 57722 rows, 62 columns


In [5]:
pd.read_sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT income_segment) as income_segments,
        COUNT(DISTINCT revol_util_bucket) as util_buckets,
        COUNT(DISTINCT credit_age_bucket) as age_buckets,
        AVG(income_to_loan_ratio) as avg_income_to_loan,
        SUM(CASE WHEN income_to_loan_ratio IS NULL THEN 1 ELSE 0 END) as nulls_in_income_loan,
        SUM(CASE WHEN overall_leverage_ratio IS NULL THEN 1 ELSE 0 END) as nulls_in_leverage,
        SUM(risk_flag_count) as total_risk_flags
    FROM loan_features
""", engine)

,total_rows,income_segments,util_buckets,age_buckets,avg_income_to_loan,nulls_in_income_loan,nulls_in_leverage,total_risk_flags
0,57722,4,4,3,7.269805,1,3132,28419


## Next Step
- Features exported to `data/features/loan_features.csv` (57,722 rows, 62 columns).
- Proceeding to `04_modelling.ipynb` for baseline logistic regression, random forest, and XGBoost models.